# SSL DAPT + LoRA Training on Qwen3 Models

This notebook demonstrates Domain-Adaptive Pre-Training (DAPT) with LoRA on Qwen3-4B or Qwen3-8B models using JSON/JSONL corpus data.

**SSL DAPT**: Self-Supervised Learning Domain-Adaptive Pre-Training continues pre-training a language model on domain-specific text using causal language modeling objective.

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets accelerate peft bitsandbytes wandb tqdm

## 2. Import Libraries

In [ ]:
import os
import json
import logging
from pathlib import Path
from typing import List, Dict, Any, Optional

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 3. Configuration

In [ ]:
# Model Configuration
MODEL_NAME = "Qwen/Qwen3-4B"  # Options: "Qwen/Qwen3-4B", "Qwen/Qwen3-8B", "Qwen/Qwen3-4B-Instruct-2507"
USE_4BIT = True  # 4-bit quantization for memory efficiency
USE_8BIT = False

# LoRA Configuration
LORA_R = 64  # LoRA rank
LORA_ALPHA = 128  # LoRA alpha (scaling factor)
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# Data Configuration
DATA_PATH = "./data"  # Path to JSON/JSONL files
TEXT_FIELD = "text"  # Field name containing text in JSON
MAX_SEQ_LENGTH = 2048

# Training Configuration
OUTPUT_DIR = "./outputs/qwen3_dapt_lora"
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03

## 4. Data Loading Utilities

In [ ]:
def load_json_corpus(data_path: str, text_field: str = "text") -> List[str]:
    """Load text from JSON/JSONL files."""
    data_path = Path(data_path)
    texts = []
    
    def process_item(item):
        if isinstance(item, str):
            return item.strip() if item.strip() else None
        if isinstance(item, dict):
            # Try specified field first
            if text_field in item:
                return str(item[text_field]).strip()
            # Try common fields
            for field in ["text", "content", "body", "message"]:
                if field in item:
                    return str(item[field]).strip()
            # Concatenate string values
            string_vals = [str(v) for v in item.values() if isinstance(v, str)]
            return " ".join(string_vals).strip() if string_vals else None
        return None
    
    def load_file(file_path):
        logger.info(f"Loading {file_path}")
        if file_path.suffix == ".jsonl":
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            data = json.loads(line)
                            text = process_item(data)
                            if text:
                                texts.append(text)
                        except json.JSONDecodeError:
                            continue
        else:  # .json
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        text = process_item(item)
                        if text:
                            texts.append(text)
                else:
                    text = process_item(data)
                    if text:
                        texts.append(text)
    
    if data_path.is_file():
        load_file(data_path)
    elif data_path.is_dir():
        for fp in sorted(data_path.glob("**/*.json")):
            load_file(fp)
        for fp in sorted(data_path.glob("**/*.jsonl")):
            load_file(fp)
    
    logger.info(f"Loaded {len(texts)} documents")
    return texts


class PackedDataset(Dataset):
    """Dataset that packs documents into sequences."""
    
    def __init__(self, texts, tokenizer, max_seq_length=2048, pack=True):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.examples = []
        
        if pack:
            self._pack_texts(texts)
        else:
            self._tokenize_texts(texts)
        
        logger.info(f"Created {len(self.examples)} examples")
    
    def _tokenize_texts(self, texts):
        for text in texts:
            if not text.strip():
                continue
            enc = self.tokenizer(
                text, truncation=True, max_length=self.max_seq_length,
                padding=False, return_tensors=None
            )
            self.examples.append({
                "input_ids": enc["input_ids"],
                "attention_mask": enc["attention_mask"]
            })
    
    def _pack_texts(self, texts):
        all_tokens = []
        eos_id = self.tokenizer.eos_token_id
        
        for text in texts:
            if not text.strip():
                continue
            tokens = self.tokenizer.encode(text, add_special_tokens=False)
            if tokens:
                all_tokens.extend(tokens)
                all_tokens.append(eos_id)
        
        for i in range(0, len(all_tokens), self.max_seq_length):
            chunk = all_tokens[i:i + self.max_seq_length]
            if len(chunk) > 10:
                self.examples.append({
                    "input_ids": chunk,
                    "attention_mask": [1] * len(chunk)
                })
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]

## 5. Create Sample Data (Optional)

In [ ]:
# Create sample data for testing (skip if you have your own data)
os.makedirs("./data", exist_ok=True)

sample_data = [
    {"text": "Medical knowledge: Hypertension is a common condition where the long-term force of blood against artery walls is high enough to cause health problems."},
    {"text": "Diabetes mellitus is a group of metabolic diseases characterized by high blood sugar levels over a prolonged period."},
    {"text": "Cardiovascular disease refers to conditions that involve narrowed or blocked blood vessels that can lead to heart attacks."},
    {"text": "The immune system is a complex network of cells and proteins that defends the body against infection."},
    {"text": "Antibiotics are medicines that fight bacterial infections in people and animals by killing bacteria or making it hard for them to grow."},
]

# Add more samples
for i in range(50):
    sample_data.append({"text": f"Sample medical document {i}: This contains domain-specific medical terminology and clinical information for DAPT training."})

with open("./data/sample_corpus.jsonl", "w") as f:
    for item in sample_data:
        f.write(json.dumps(item) + "\n")

print(f"Created {len(sample_data)} sample documents")

## 6. Load Model and Tokenizer

In [ ]:
# Quantization config
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=False,
    )
elif USE_8BIT:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if not bnb_config else None,
)

# Prepare for k-bit training
if bnb_config:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )

print(f"Model loaded: {MODEL_NAME}")
print(f"Model dtype: {model.dtype}")

## 7. Apply LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## 8. Prepare Dataset

In [ ]:
# Load corpus
texts = load_json_corpus(DATA_PATH, TEXT_FIELD)
print(f"Loaded {len(texts)} documents")

# Preview first document
if texts:
    print(f"\nFirst document preview:\n{texts[0][:200]}...")

In [ ]:
# Create dataset
train_dataset = PackedDataset(
    texts,
    tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    pack=True,  # Pack short sequences together
)

print(f"Dataset size: {len(train_dataset)} examples")

## 9. Training

In [ ]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal language modeling
)

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=500,
    save_total_limit=3,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    max_grad_norm=0.3,
    seed=42,
    report_to="none",  # Set to "wandb" to use W&B
    remove_unused_columns=False,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

In [ ]:
# Start training
print("Starting SSL DAPT + LoRA training...")
trainer.train()

## 10. Save Model

In [ ]:
# Save model and tokenizer
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

# Save LoRA adapter separately
lora_output_dir = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(lora_output_dir)

print(f"Model saved to: {OUTPUT_DIR}")
print(f"LoRA adapter saved to: {lora_output_dir}")

## 11. Test Inference

In [ ]:
def generate_response(prompt: str, max_new_tokens: int = 100):
    """Generate response from the model."""
    messages = [{"role": "user", "content": prompt}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return response

In [ ]:
# Test the model
test_prompts = [
    "Who are you?",
    "What is hypertension?",
    "Explain diabetes mellitus.",
]

for prompt in test_prompts:
    print(f"\n{'='*50}")
    print(f"Prompt: {prompt}")
    print(f"{'='*50}")
    response = generate_response(prompt)
    print(f"Response: {response}")

## 12. Load Saved Model (Optional)

In [ ]:
# To load the saved model later:
from peft import PeftModel

def load_dapt_model(base_model_name: str, lora_adapter_path: str):
    """Load the DAPT-trained model with LoRA adapter."""
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    
    # Load LoRA adapter
    model = PeftModel.from_pretrained(base_model, lora_adapter_path)
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name,
        trust_remote_code=True,
    )
    
    return model, tokenizer

# Example:
# model, tokenizer = load_dapt_model("Qwen/Qwen3-4B", "./outputs/qwen3_dapt_lora/lora_adapter")